# CellTransformer on your own dataset

This notebook runs the native CellTransformer commands step by step:
`celltransformer-preprocess` → `celltransformer-train` → `celltransformer-extract` → `celltransformer-smooth` → `celltransformer-cluster`.
Training is configured with Hydra: every setting in `config/` can be overridden from the command
line, and this notebook builds those overrides from the values you choose.

As a worked example it uses five sections of the Allen 1 MERFISH data (the same files as `01_allen_dataset.ipynb`),
downloaded in Step 1. Cells and variables marked **REPLACE WITH YOUR DATA** are where you
point it at your own files.

**Requirements:** Linux x86_64 and an NVIDIA GPU with bfloat16 support. Two environments,
both with the package installed (`pip install -e .` from the repository):
- a training environment (this notebook's kernel), and
- a RAPIDS environment with cuML, used for smoothing and clustering.

Notebook support and HTML reporting are included in the installed CellTransformer package.
No companion Python files are needed next to this notebook.

**Input contract.** Every later step reads the standard input written in Step 3, one pair per section:
an `.h5ad` of raw counts (cells × genes) in `layers['raw']` and a paired `.parquet` of the same cells, in the same order,
with columns `cell_id`, `section`, `x` and `y` (µm), plus `cell_type` when `USE_CELL_TYPE=True`.
Row *i* of the metadata must describe row *i* of the count matrix. Cell-type annotations and
`taxonomy.yaml` are not required when `USE_CELL_TYPE=False`; the native model then uses the
same internal type label for all cells, without inferring or inventing biological cell types.

## Step 1: set paths and dataset columns
● TIMING <1 min, plus the first download (~9.3 GB)

Set `WORK_DIR` and `RAPIDS_BIN` below. The next cell downloads the example Allen files (~9.3 GB) into
`DATA_ROOT/data/raw/`; skip it if you use your own data.

**REPLACE WITH YOUR DATA:** everything in the first block below. `SECTIONS` selects which tissue
sections to use; `VALID_SECTIONS` are held out entirely from training and used only for the
validation loss (split by section, not by cell, so validation measures generalization to unseen tissue).

For data without cell-type annotations, set `USE_CELL_TYPE=False` and `CELL_TYPE_COL=None`.
Keep `USE_CELL_TYPE=True` for the unchanged Allen example. Set `DROP_GENE_PREFIX=None` if
there are no control probes to remove.

Use a new `WORK_DIR` when changing the input data, cell-type mode or training settings.
The settings records below prevent incompatible prepared files, checkpoints or embeddings from
being silently reused. Existing results are never deleted. To resume this updated notebook,
reuse its work directory and the same settings. A directory created by an older notebook has
no settings records and should be kept as an archive; start the updated workflow in a new directory.

In [ ]:
import sys
import json
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import yaml

from celltransformer.notebooks import helper

#REPLACE WITH YOUR DATA: raw count matrix (.h5ad) and per-cell metadata (.csv or .parquet)
#(for the example, the next cell downloads the Allen files into DATA_ROOT/data/raw/)
DATA_ROOT = Path("~/allen_data").expanduser().resolve()
RAW_H5AD = DATA_ROOT / "data/raw/expression_raw.h5ad"
RAW_METADATA = DATA_ROOT / "data/raw/cells_ccf.csv"

#REPLACE WITH YOUR DATA: metadata column names
CELL_ID_COL = "cell_label"            #matches the .h5ad obs_names
SECTION_COL = "brain_section_label"
USE_CELL_TYPE = True                 #False: run the native model without cell-type annotations
CELL_TYPE_COL = "subclass"           #None when USE_CELL_TYPE=False
X_COL = "x_reconstructed"
Y_COL = "y_reconstructed"
COORD_TO_UM = 1000.0                  #multiply coordinates by this to get µm (Allen stores mm)

#REPLACE WITH YOUR DATA: genes to drop (here, MERFISH blank control probes), matched on this var column
GENE_SYMBOL_COL = "gene_symbol"
DROP_GENE_PREFIX = "Blank"          #None: skip this prefix filter; GENE_SYMBOL_COL is then unused

#REPLACE WITH YOUR DATA: sections to use and sections held out for validation
SECTIONS = [f"C57BL6J-638850.{n}" for n in [28, 29, 30, 31, 32]]
VALID_SECTIONS = ["C57BL6J-638850.30"]

#working directory for everything this notebook writes
WORK_DIR = Path("~/ct2_runs/custom").expanduser().resolve()

#bin/ directories of the two environments: this kernel's, and the RAPIDS one (set this)
DEV_BIN = Path(sys.executable).parent
#REPLACE WITH YOUR ENV: bin/ of celltransformer-rapids; run `conda env list` and add /bin to its path
RAPIDS_BIN = Path("/path/to/rapids-env/bin").resolve()

#derived paths
INPUT_DIR = WORK_DIR / "input"
SECTIONS_DIR = WORK_DIR / "sections"
SMOOTHED_DIR = WORK_DIR / "sections_smoothed"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
DOMAINS_DIR = WORK_DIR / "domains"
TAXONOMY = WORK_DIR / "taxonomy.yaml"
TRAIN_LOG = WORK_DIR / "train.log"
CHECKPOINT_OVERRIDE = None          #optional path to another compatible checkpoint; use a new WORK_DIR
for d in [INPUT_DIR, SECTIONS_DIR, CHECKPOINT_DIR, DOMAINS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def _file_identity(path):
    path = Path(path).expanduser().resolve()
    stat = path.stat()
    return {"path": str(path), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}

def _check_run_settings(name, settings, outputs_exist=False):
    """Refuse incompatible cached outputs; keep previous results intact."""
    record = WORK_DIR / f"{name}_settings.json"
    if record.exists():
        if json.loads(record.read_text()) != settings:
            raise ValueError(f"{name} settings changed in {WORK_DIR}. Use a new WORK_DIR; existing results are preserved.")
    elif outputs_exist:
        raise ValueError(f"Existing {name} outputs have no settings record. Use a new WORK_DIR; existing results are preserved.")
    else:
        temporary = record.with_suffix(".json.tmp")
        temporary.write_text(json.dumps(settings, indent=2, sort_keys=True) + "\n")
        temporary.replace(record)

assert isinstance(USE_CELL_TYPE, bool), "USE_CELL_TYPE must be True or False"
assert not USE_CELL_TYPE or CELL_TYPE_COL, "Set CELL_TYPE_COL when USE_CELL_TYPE=True"
#Source file identity is recorded after the optional download, in Step 3 or Step 5.
def _check_input_settings():
    settings = {
        "raw_expression": _file_identity(RAW_H5AD), "raw_metadata": _file_identity(RAW_METADATA),
        "cell_id_column": CELL_ID_COL, "section_column": SECTION_COL,
        "use_cell_type": USE_CELL_TYPE, "cell_type_column": CELL_TYPE_COL if USE_CELL_TYPE else None,
        "x_column": X_COL, "y_column": Y_COL, "coordinate_to_um": COORD_TO_UM,
        "gene_symbol_column": GENE_SYMBOL_COL if DROP_GENE_PREFIX else None,
        "drop_gene_prefix": DROP_GENE_PREFIX or None, "sections": list(SECTIONS),
    }
    existing = any(directory.exists() and any(directory.iterdir())
                   for directory in (INPUT_DIR, SECTIONS_DIR, CHECKPOINT_DIR, SMOOTHED_DIR, DOMAINS_DIR))
    _check_run_settings("input", settings, outputs_exist=existing)


In [ ]:
%%time
#example data only: download the raw Allen files (~9.3 GB) and verify their checksums; verified files are skipped
helper.download_allen(DATA_ROOT)

## Step 2: check the environment
● TIMING ~1 min

Lists the visible GPUs, checks that PyTorch can use the GPU in this environment, and checks that the
RAPIDS environment in `RAPIDS_BIN` has the smoothing and clustering commands and a working cuML/cuDF.
Do not continue until it reports `ENVIRONMENT CHECK: PASSED`.

In [ ]:
%%time
helper.check_environment(DEV_BIN, RAPIDS_BIN)

## Step 3: write the standard input
● TIMING ~5 min for the example

**REPLACE WITH YOUR DATA:** this step is dataset-specific. For the Allen example it:
1. writes the cell-type taxonomy from the **full** metadata,
2. keeps the chosen sections and annotated cells,
3. matches metadata rows to count rows by exact cell ID,
4. drops blank control probes, and
5. converts coordinates to µm.

For data without annotations, `USE_CELL_TYPE=False` skips the taxonomy and annotation filter;
all selected cells with matching counts are retained. No `cell_type` column is written.
If you prepare the same file pairs by another route, use a new work directory and adapt the
input settings record to describe that preparation before continuing.

▲ **CRITICAL STEP** When cell types are enabled, the taxonomy is an ordered list; a cell type's position in it is its integer label.
Build it from the complete dataset, not the subset, so the labels mean the same thing across runs and
datasets. `celltransformer-train` and `celltransformer-extract` must use the same file.

▲ **CRITICAL STEP** Keep raw counts. `celltransformer-preprocess` applies log2(1 + x) in Step 4; do not
normalize or log-transform beforehand.

The counts are read and written one section at a time (`input/{section}.h5ad` + `.parquet`), so only
one section's counts are in memory; the metadata for all cells stays loaded. Rerunning the second cell
skips sections already written, so the step can continue after a crash.

In [ ]:
%%time
_check_input_settings()
#Only request a cell-type column when it will be used by the model.
columns = list(dict.fromkeys([CELL_ID_COL, SECTION_COL, X_COL, Y_COL]
                            + ([CELL_TYPE_COL] if USE_CELL_TYPE else [])))
if RAW_METADATA.suffix == ".parquet":
    metadata = pd.read_parquet(RAW_METADATA, columns=columns)
else:
    metadata = pd.read_csv(RAW_METADATA, usecols=columns, dtype={CELL_ID_COL: str})
assert metadata[CELL_ID_COL].notna().all(), "metadata cell IDs cannot be missing"
metadata[CELL_ID_COL] = metadata[CELL_ID_COL].astype(str)
assert metadata[CELL_ID_COL].is_unique, "metadata cell IDs must be unique"

taxonomy = None
if USE_CELL_TYPE:
    #Preserve the Allen example's taxonomy and annotated-cell selection.
    taxonomy = sorted(str(t) for t in metadata[CELL_TYPE_COL].dropna().unique())
    assert taxonomy, "no cell-type annotations found; check CELL_TYPE_COL or set USE_CELL_TYPE=False"
    taxonomy_text = yaml.safe_dump(taxonomy, allow_unicode=True)
    if not TAXONOMY.exists() or TAXONOMY.read_text() != taxonomy_text:
        TAXONOMY.write_text(taxonomy_text)
    print(f"taxonomy: {len(taxonomy)} cell types -> {TAXONOMY}")

metadata = metadata[metadata[SECTION_COL].isin(SECTIONS)]
if USE_CELL_TYPE:
    metadata = metadata.dropna(subset=[CELL_TYPE_COL])

#Match rows by exact cell ID, then retain file order for backed reads.
counts = ad.read_h5ad(RAW_H5AD, backed="r")
assert counts.obs_names.is_unique, "expression cell IDs must be unique"
assert counts.var_names.is_unique, "gene identifiers must be unique"
count_rows = pd.Index(counts.obs_names).get_indexer(metadata[CELL_ID_COL])
has_counts = count_rows >= 0
print(f"{has_counts.sum():,} selected cells with counts; {(~has_counts).sum():,} without counts dropped")
metadata = metadata[has_counts]
count_rows = count_rows[has_counts]
assert len(metadata), "no selected cells have matching counts"
order = np.argsort(count_rows)
metadata = metadata.iloc[order].reset_index(drop=True)
count_rows = count_rows[order]

#Set DROP_GENE_PREFIX=None to disable this additional prefix filter.
keep_genes = np.ones(counts.n_vars, dtype=bool)
if DROP_GENE_PREFIX:
    assert GENE_SYMBOL_COL in counts.var, "set GENE_SYMBOL_COL or disable DROP_GENE_PREFIX"
    keep_genes = ~counts.var[GENE_SYMBOL_COL].astype(str).str.startswith(DROP_GENE_PREFIX).to_numpy()
genes = counts.var[keep_genes]
assert len(genes), "no genes remain after filtering"

standard_metadata = pd.DataFrame({
    "cell_id": metadata[CELL_ID_COL].to_numpy(),
    "section": metadata[SECTION_COL].astype(str).to_numpy(),
    "x": metadata[X_COL].to_numpy() * COORD_TO_UM,
    "y": metadata[Y_COL].to_numpy() * COORD_TO_UM,
})
if USE_CELL_TYPE:
    standard_metadata["cell_type"] = metadata[CELL_TYPE_COL].astype(str).to_numpy()
print(f"{len(standard_metadata):,} cells x {len(genes)} genes in {standard_metadata['section'].nunique()} sections")


In [ ]:
%%time
#one input pair per section, reading only that section's counts
for section, rows in standard_metadata.groupby("section").indices.items():
    h5ad_path = INPUT_DIR / f"{helper.safe_section(section)}.h5ad"
    parquet_path = h5ad_path.with_suffix(".parquet")

    #the parquet is written last, so it marks a finished section (a rerun after a crash skips it)
    if parquet_path.exists():
        print("exists, skipping:", parquet_path.name)
        continue

    section_metadata = standard_metadata.iloc[rows].reset_index(drop=True)
    section_raw = counts[count_rows[rows]].to_memory()

    #row alignment check before writing
    assert (section_raw.obs_names == section_metadata["cell_id"]).all()

    #raw counts only in layers['raw'], which preprocess transforms into .X (so .X stays empty here)
    section_counts = ad.AnnData(
        obs=pd.DataFrame(index=section_metadata["cell_id"].to_numpy()),
        var=genes,
        layers={"raw": section_raw.X[:, keep_genes].astype(np.float32)},
    )
    section_counts.write_h5ad(h5ad_path)
    section_metadata.to_parquet(parquet_path, index=False)
    print(f"wrote {h5ad_path.name}: {len(section_metadata):,} cells")

counts.file.close()
standard_metadata.groupby("section").size()

## Step 4: log-transform the counts
● TIMING ~2 min for the example

Step 3 is specific to the dataset; from here on, every step runs the same for any dataset.
`celltransformer-preprocess` runs once per section written in Step 3, so it loads one section at a time. It writes an
`.h5ad`/`.parquet` pair per section with log2(1 + x) of the raw counts in `.X`. Section names have `.` and
spaces replaced by `_` in the filenames. Sections already in `SECTIONS_DIR` are skipped.

In [ ]:
%%time
#one preprocess call per section (only that section in memory); finished sections are skipped
for parquet_path in sorted(INPUT_DIR.glob("*.parquet")):
    if (SECTIONS_DIR / f"section_{parquet_path.stem}.parquet").exists():
        print("exists, skipping:", parquet_path.stem)
        continue
    helper.run("celltransformer-preprocess", parquet_path.with_suffix(".h5ad"),
        "--output-dir", SECTIONS_DIR,
        "--output-stem", "section",
        "--section-col", "section",
        "--transform", "log2p1",
        cwd=WORK_DIR);

## Step 5: check the per-section files
● TIMING ~1 min

▲ **CRITICAL STEP** Check exact cell-ID alignment, identical gene identities and order, finite
coordinates and the required columns in every section. When cell types are enabled, labels must
belong to the shared taxonomy. The check also samples up to 1,000 cells across each section and
compares `.X` with `log2(1 + layers["raw"])`; this verifies the transformation at the sampled rows,
not every expression value. Fix any failure before training.

After a kernel restart, rerun Step 1 and then start here; Step 3 does not need rerunning.

In [ ]:
%%time
_check_input_settings()
#A restarted kernel can resume here after Step 1, including without a taxonomy.
taxonomy = yaml.safe_load(TAXONOMY.read_text()) if USE_CELL_TYPE else None
if USE_CELL_TYPE:
    assert isinstance(taxonomy, list) and taxonomy, "taxonomy must be a nonempty ordered list"
    assert len(taxonomy) == len(set(taxonomy)), "taxonomy contains duplicate labels"

section_files = sorted(SECTIONS_DIR.glob("*.h5ad"))
assert section_files, "no prepared section files; run Steps 3 and 4 first"
first_section = ad.read_h5ad(section_files[0], backed="r")
expected_genes = first_section.var_names.copy()
first_section.file.close()

def check_section(h5ad_path):
    """Check identities and order, coordinates, optional types and sampled counts."""
    expression = ad.read_h5ad(h5ad_path, backed="r")
    try:
        section_metadata = pd.read_parquet(h5ad_path.with_suffix(".parquet"))
        required = {"cell_id", "section", "x", "y"} | ({"cell_type"} if USE_CELL_TYPE else set())
        assert required <= set(section_metadata), f"{h5ad_path.name}: missing metadata columns {required - set(section_metadata)}"
        ids = section_metadata["cell_id"]
        rows_match = (expression.n_obs == len(ids) and ids.notna().all() and ids.is_unique
                      and np.array_equal(expression.obs_names.to_numpy(), ids.astype(str).to_numpy()))
        genes_match = expression.var_names.is_unique and expression.var_names.equals(expected_genes)
        coordinates_ok = np.isfinite(section_metadata[["x", "y"]].to_numpy(dtype=float)).all()
        sections_ok = section_metadata["section"].notna().all() and section_metadata["section"].nunique() == 1
        types_ok = (not USE_CELL_TYPE or (section_metadata["cell_type"].notna().all()
                    and section_metadata["cell_type"].astype(str).isin(taxonomy).all()))
        assert expression.n_obs > 0 and expression.n_vars > 0, f"{h5ad_path.name}: empty expression matrix"
        assert "raw" in expression.layers, f"{h5ad_path.name}: raw counts are needed to check the transformation"
        sample_rows = np.linspace(0, expression.n_obs - 1, min(1000, expression.n_obs), dtype=int)
        sample = expression.X[sample_rows]
        raw = expression.layers["raw"][sample_rows]
        sample = sample.toarray() if hasattr(sample, "toarray") else np.asarray(sample)
        raw = raw.toarray() if hasattr(raw, "toarray") else np.asarray(raw)
        counts_ok = np.isfinite(raw).all() and (raw >= 0).all()
        transformed_ok = (counts_ok and np.isfinite(sample).all()
                          and np.allclose(sample, np.log2(1.0 + raw), rtol=1e-5, atol=1e-6))
        return {"file": h5ad_path.name, "cells": expression.n_obs, "genes": expression.n_vars,
                "cell_ids_aligned": bool(rows_match), "gene_order_matches": bool(genes_match),
                "coordinates_ok": bool(coordinates_ok), "section_ok": bool(sections_ok),
                "types_ok": bool(types_ok), "sample_log2_matches_raw": bool(transformed_ok)}
    finally:
        expression.file.close()

summary = pd.DataFrame([check_section(path) for path in section_files])
checks = ["cell_ids_aligned", "gene_order_matches", "coordinates_ok", "section_ok",
          "types_ok", "sample_log2_matches_raw"]
assert summary[checks].all().all(), "prepared input check failed:\n" + summary.to_string(index=False)
N_GENES = int(summary["genes"].iloc[0])
summary


## Step 6: choose the design
● TIMING <1 min

These are the choices you may tune. The values below match the full Allen run in `01_allen_dataset.ipynb`,
except for the shorter training budget.

- **Neighborhood:** `PATCH_SIZE` is the side of the square window around each cell, in the same units as
  `x`/`y` (µm here). Pick it for the physical scale of the structure you study.
- **Model size:** `MODEL_DIM` is the width, the length of the vector each cell becomes inside the
  transformer and of the final embedding. `NUM_HEADS` must divide it evenly. Larger models capture more
  detail but need more memory and data.
- **Training:** `EPOCHS`, learning rate and warmup. `BATCH_SIZE` is centers per GPU and `MAX_TOKENS` caps
  the total neighbor cells per batch; lower `MAX_TOKENS` if you run out of GPU memory.
- **GPUs:** `DEVICES` lists GPU indices. The batch is per GPU, so more GPUs means a larger effective
  batch; use `ACCUMULATE` to compensate when comparing runs on different GPU counts.

The number of genes and, when enabled, cell types are set by your data (Steps 3 and 5).
Without annotations, the native model uses one shared internal type label. Changing these settings
or retraining requires a new `WORK_DIR` so previous checkpoints and outputs remain intact.

In [ ]:
#neighborhood (coordinate units, µm here)
PATCH_SIZE = 85.0

#model
MODEL_DIM = 384
NUM_HEADS = 8
ENCODER_DEPTH = 4
DECODER_DEPTH = 4
FFN_DIM = 384
DROPOUT = 0.0

#training
EPOCHS = 1
LEARNING_RATE = 3e-4
WARMUP_STEPS = 500
BATCH_SIZE = 256
MAX_TOKENS = 16384
ACCUMULATE = 1
DEVICES = [0]                         #GPU indices

#set by the data
N_TYPES = len(taxonomy) if USE_CELL_TYPE else 1

assert MODEL_DIM % NUM_HEADS == 0, "NUM_HEADS must divide MODEL_DIM"
print(f"expression_dim={N_GENES}, cell_type_cardinality={N_TYPES}, head size={MODEL_DIM // NUM_HEADS}")

In [ ]:
neighbors = helper.neighbor_counts(section_files[0].with_suffix(".parquet"), PATCH_SIZE)
print(f"neighbors per cell at PATCH_SIZE={PATCH_SIZE}: median {np.median(neighbors):.0f}, "
      f"5th-95th percentile {np.percentile(neighbors, 5):.0f}-{np.percentile(neighbors, 95):.0f}, "
      f"{(neighbors == 0).mean():.1%} with none")

▲ **CRITICAL STEP** If most cells have zero neighbors, or thousands, `PATCH_SIZE` is probably in the
wrong units for your coordinates (for example mm vs µm). Nothing downstream checks this.

## Step 7: train
● TIMING depends on data size, epochs and GPUs (see Timing)

`celltransformer-train` composes its configuration from `config/train.yaml` and the
`data`, `model` and `optimization` groups, then applies the overrides built below. Every value set here
replaces the default in those YAML files for this run only; the files themselves are not changed.
Training runs in the background, logging to `train.log`. W&B runs offline, so no account is needed.
If `checkpoints/` already has a finished run, this cell does nothing.

In [ ]:
_check_input_settings()
#validation sections are excluded from training automatically
valid_patterns = ",".join(f'"*{helper.safe_section(s)}.h5ad"' for s in VALID_SECTIONS)

overrides = [
    #data
    f"data.data_dir='{SECTIONS_DIR}/'",
    'data.train_anndata_patterns=["*.h5ad"]',
    f"data.valid_anndata_patterns=[{valid_patterns}]",
    "data.tissue_section_colname=section",
    f"data.patch_size=[{PATCH_SIZE},{PATCH_SIZE}]",
    f"data.batch_size={BATCH_SIZE}",
    f"data.max_tokens={MAX_TOKENS}",
    #model
    f"model.use_cell_type={str(USE_CELL_TYPE).lower()}",
    f"model.expression_dim={N_GENES}",
    f"model.cell_type_cardinality={N_TYPES}",
    f"model.model_dim={MODEL_DIM}",
    f"model.num_heads={NUM_HEADS}",
    f"model.encoder_depth={ENCODER_DEPTH}",
    f"model.decoder_depth={DECODER_DEPTH}",
    f"model.ffn_dim={FFN_DIM}",
    f"model.dropout={DROPOUT}",
    #optimization
    f"optimization.lr={LEARNING_RATE}",
    f"optimization.schedule_warmup={WARMUP_STEPS}",
    f"optimization.max_epochs={EPOCHS}",
    f"optimization.accumulate_grad_batches={ACCUMULATE}",
    #machine and outputs
    f"devices=[{','.join(str(d) for d in DEVICES)}]",
    f"checkpoint_dir='{CHECKPOINT_DIR}'",
    "wandb_project=celltransformer-custom",
    f"output='{WORK_DIR / 'train_results.jsonl'}'",
    f"hydra.run.dir='{WORK_DIR / 'hydra'}'",
]
if USE_CELL_TYPE:
    overrides += ["data.cell_type_colname=cell_type", f"data.cell_type_yaml='{TAXONOMY}'"]
else:
    overrides += ["data.cell_type_colname=null", "data.cell_type_yaml=null"]
_check_run_settings("training", {
    "overrides": overrides,
    "prepared_inputs": [_file_identity(path) for h5ad in section_files
                        for path in (h5ad, h5ad.with_suffix(".parquet"))],
    "taxonomy": _file_identity(TAXONOMY) if USE_CELL_TYPE else None,
}, outputs_exist=any(CHECKPOINT_DIR.iterdir()))
print("celltransformer-train \\\n    " + " \\\n    ".join(overrides))

In [ ]:
helper.start_training(DEV_BIN, overrides, WORK_DIR, TRAIN_LOG, CHECKPOINT_DIR)

■ **PAUSE POINT** Training runs outside the notebook: it continues if you close the notebook, restart or shut
down the kernel, or disconnect from the machine. Rerun Steps 1, 5 and 6 when you come back,
then Step 8.

## Step 8: follow training
● TIMING <1 min (rerun as often as you like)

Shows whether training is alive and which finished run Step 9 will use. While training runs, or if no run has
finished yet, it also shows the end of the log. Each run gets a
timestamped folder under `checkpoints/`; `last.ckpt` is the final state.

▲ **CRITICAL STEP** Before extracting embeddings, the validation loss (`valid/nll`) should have stopped
decreasing. A loss still falling means more epochs are needed; a validation loss rising while training
loss falls means overfitting.

In [ ]:
helper.show_training(WORK_DIR, TRAIN_LOG, CHECKPOINT_DIR)

## Step 9: extract embeddings
● TIMING minutes per section, depending on section size

Passes every cell's neighborhood through the trained encoder and writes one `*_embeddings.npz` per
section next to its `.h5ad`, with `embeddings` (cells × `MODEL_DIM`) and `has_neighbors`.
By default it uses the most recent finished run. To select another compatible checkpoint, set
`CHECKPOINT_OVERRIDE` in Step 1. Use a new `WORK_DIR` and prepare its inputs before extracting from
a different checkpoint: existing embeddings are skipped by the native command, not overwritten.
The settings check below rejects incompatible cached embeddings and downstream outputs.

▲ **CRITICAL STEP** The patch size, ordered gene panel and, when used, cell-type taxonomy must
match training. These are reused from Steps 5 and 6 here; after a restart, rerun those cells first.
The checkpoint determines whether extraction needs cell types. A mismatch with `USE_CELL_TYPE`
stops the cell before extraction.

In [ ]:
%%time
from celltransformer.notebooks.reporting import read_use_cell_type

_check_input_settings()
if CHECKPOINT_OVERRIDE is None:
    run = helper.finished_run(CHECKPOINT_DIR)
    assert run is not None, "training has not finished yet (no finished run in CHECKPOINT_DIR); see Step 8"
    CHECKPOINT = run / "last.ckpt"
else:
    CHECKPOINT = Path(CHECKPOINT_OVERRIDE).expanduser().resolve()
assert CHECKPOINT.is_file(), f"checkpoint not found: {CHECKPOINT}"
checkpoint_use_cell_type = read_use_cell_type(CHECKPOINT)
assert checkpoint_use_cell_type == USE_CELL_TYPE, "checkpoint cell-type mode differs from USE_CELL_TYPE; use matching inputs in a new WORK_DIR"
print("checkpoint:", CHECKPOINT)

_check_run_settings("extraction", {
    "checkpoint": _file_identity(CHECKPOINT), "use_cell_type": checkpoint_use_cell_type,
    "patch_size": PATCH_SIZE, "device": f"cuda:{DEVICES[0]}",
    "prepared_inputs": [_file_identity(path) for h5ad in section_files
                        for path in (h5ad, h5ad.with_suffix(".parquet"))],
    "taxonomy": _file_identity(TAXONOMY) if checkpoint_use_cell_type else None,
}, outputs_exist=(any(SECTIONS_DIR.glob("*_embeddings.npz"))
                  or (SMOOTHED_DIR.exists() and any(SMOOTHED_DIR.iterdir()))
                  or any(DOMAINS_DIR.iterdir())))
type_args = (["--cell-type-yaml", TAXONOMY, "--cell-type-colname", "cell_type"]
             if checkpoint_use_cell_type else [])
for h5ad_path in section_files:
    helper.run("celltransformer-extract", h5ad_path,
        "--checkpoint-path", CHECKPOINT,
        *type_args,
        "--section-colname", "section",
        "--patch-size", PATCH_SIZE,
        "--device", f"cuda:{DEVICES[0]}",
        cwd=WORK_DIR, quiet=True)


## Step 10: smooth the embeddings spatially
● TIMING ~2 min for the example

`celltransformer-smooth` replaces each cell's embedding with a Gaussian-weighted average of its
`N_NEIGHBORS` nearest neighbors in the same section (kernel width `SIGMA`, in µm). Sections are smoothed
independently, so no averaging crosses tissue boundaries. Runs in the RAPIDS environment, on the first GPU
in `DEVICES`.

It reads every `*_embeddings.npz` in `SECTIONS_DIR` and writes files of the same name, plus a copy of each
`.parquet`, to `SMOOTHED_DIR`. That folder has the same layout as `SECTIONS_DIR`, so Step 11 can cluster either.

▲ **CRITICAL STEP** `SIGMA` is in the same units as `x`/`y`. It is the standard deviation of the Gaussian;
the kernel's full width at half maximum (FWHM) is 2.355 × `SIGMA`. `SIGMA` = 36 µm gives a FWHM of ~85 µm,
the same as the neighborhood (`PATCH_SIZE`, Step 6).

In [ ]:
%%time
#smoothing kernel (coordinate units, µm here) and neighbors per cell
SIGMA = 36.0
N_NEIGHBORS = 500

helper.run("celltransformer-smooth", SECTIONS_DIR, SMOOTHED_DIR,
    "--gaussian-kernel", SIGMA,
    "--n-neighbors", N_NEIGHBORS,
    "--spatial-x-col", "x",
    "--spatial-y-col", "y",
    bin_dir=RAPIDS_BIN, cwd=WORK_DIR, gpu=DEVICES[0]);

## Step 11: cluster into spatial domains
● TIMING ~1 min per k per embedding set for the example

GPU k-means on all sections' embeddings together, once for each k in `KS`, first on the unsmoothed embeddings
(`SECTIONS_DIR`) and then on the smoothed ones (`SMOOTHED_DIR`). k is decoupled from training:
you can rerun this step with other values without retraining. Runs in the RAPIDS environment.

The clustering runs on the first GPU in `DEVICES` (`CUDA_VISIBLE_DEVICES`), since
`celltransformer-cluster` otherwise uses GPU 0.

In [ ]:
%%time
KS = [20, 50]

#output tag for each embedding set
EMBEDDING_SETS = {"custom": SECTIONS_DIR, "custom_smoothed": SMOOTHED_DIR}

k_args = [arg for k in KS for arg in ["--ks", k]]
for tag, embeddings_dir in EMBEDDING_SETS.items():
    helper.run("celltransformer-cluster",
        "--embeddings-dirs", embeddings_dir,
        "--output-base", DOMAINS_DIR,
        "--tag", tag,
        *k_args,
        "--x-col", "x",
        "--y-col", "y",
        "--section-col", "section",
        bin_dir=RAPIDS_BIN, cwd=WORK_DIR, gpu=DEVICES[0]);

## Step 12: view the domains
● TIMING <1 min

Labels are saved as `domains/{tag}_kmeans_k{k}_labels.npy` (one label per cell, in section-file order),
and one plot per section in `domains/{tag}_cluster_plots_k{k}/`, for `tag` = `custom` (unsmoothed) and
`custom_smoothed`. Colors are assigned independently in each run, so the same color does not mean the same
domain across plots.

In [ ]:
helper.show_domain_plots(DOMAINS_DIR, EMBEDDING_SETS, KS, n_show=1)

## Step 13: generate the HTML report (optional)
● TIMING depends on the number of cells, sections and clustering resolutions

Run this after Steps 9–11 have finished. It reads the saved assignments and writes a new,
self-contained `review.html` and `results.zip` under `WORK_DIR/html_reports/`. It does not retrain,
extract, smooth or cluster again. Training curves are recovered from the selected checkpoint's local
W&B history; missing values are reported rather than replaced with batch losses.

The report contains training diagnostics and maps for every section and k. Anatomical reference
annotations are optional and can be shown on reference maps whether or not cell types were used.
When the checkpoint used cell types and reference annotations are supplied, the report also shows
reference–domain correspondence heatmaps: Pearson correlations between cell-type proportions,
with a fixed −1 to +1 color scale. With cell types disabled, these heatmaps are omitted;
training curves, domain maps and any supplied reference maps are retained. Reference regions and discovered domains
need not have the same number of labels. The report has no separate composition-matrix panels.

The reporting functions are included in the installed CellTransformer package.
See the [reporting guide](../docs/notebook_html_report.md).
Section names come from the data. Set the report title, coordinate units and optional reference
columns below. To omit reference comparison, set `REFERENCE_COLUMNS = {}`. Reference rows are joined
by exact cell ID; a missing annotation excludes that cell from correspondence calculations, not from
training or the saved domain assignments. Colors are consistent across sections within each method
and k, but do not imply matched identities across methods or the reference.


In [ ]:
from celltransformer.notebooks.reporting import report_from_notebook, read_reference_table, read_training_history, read_use_cell_type, show_report

# REPLACE WITH YOUR DATA: display title, coordinate units and optional reference columns.
REPORT_NAME = WORK_DIR.name
REPORT_COORD_UNITS = "µm"
REFERENCE_FILE = RAW_METADATA
REFERENCE_ID_COLUMN = CELL_ID_COL
# The example uses CCF substructures. For other data, replace this mapping with
# {"Reference name": "your_annotation_column"}, or {} if there is no reference.
REFERENCE_COLUMNS = {"Reference regions": "parcellation_substructure"}  #set {} if no anatomical reference
REFERENCE_FOR_K = None  # a single reference is compared at every k in KS
REFERENCE_DISPLAY_NAMES = {"Reference regions": "Reference"}

# Optional annotation-based enlargement; leave None for whole-section maps only.
# See the reporting guide linked above for explicit crop bounds and other reference layouts.
REPORT_CROP = None
REPORT_CROP_NAME = "Enlargement"
REPORT_BOUNDARY_GRID = None
REPORT_BOUNDARY_DISTANCE = None


In [ ]:
# Read the completed checkpoint's local epoch averages; no W&B login is needed.
report_use_cell_type = read_use_cell_type(CHECKPOINT)
report_training = read_training_history(CHECKPOINT, expected_epochs=EPOCHS)
report_reference = read_reference_table(REFERENCE_FILE, REFERENCE_ID_COLUMN, REFERENCE_COLUMNS)

report_notes = report_training["notes"] + [
    f"{len(VALID_SECTIONS)} whole sections were withheld during training; clustering used all sections jointly.",
]
if report_use_cell_type and report_reference is not None:
    report_notes.append("Reference correspondence is an exploratory comparison of cell-type composition, not an independent held-out clustering test.")
report_html = report_from_notebook(
    work_dir=WORK_DIR,
    embedding_sets=EMBEDDING_SETS,
    domains_dir=DOMAINS_DIR,
    ks=KS,
    dataset_name=REPORT_NAME,
    coordinate_units=REPORT_COORD_UNITS,
    n_genes=N_GENES,
    use_cell_type=report_use_cell_type,
    method_names={tag: "CellTransformer (+smoothing)" if Path(folder).resolve() == SMOOTHED_DIR.resolve()
                  else "CellTransformer (−smoothing)" for tag, folder in EMBEDDING_SETS.items()},
    reference_metadata=report_reference,
    reference_id_column=REFERENCE_ID_COLUMN,
    reference_columns=REFERENCE_COLUMNS,
    reference_for_k=REFERENCE_FOR_K if report_reference is not None else None,
    reference_display_names=REFERENCE_DISPLAY_NAMES,
    training_history=report_training["history"],
    notes=report_notes,
    provenance={"training": report_training["provenance"], "validation_sections": list(VALID_SECTIONS),
                "use_cell_type": report_use_cell_type, "patch_size": PATCH_SIZE, "smoothing_sigma": SIGMA, "smoothing_neighbors": N_NEIGHBORS},
    crop_selection=REPORT_CROP if report_reference is not None else None,
    crop_name=REPORT_CROP_NAME,
    boundary_grid_size=REPORT_BOUNDARY_GRID,
    boundary_max_distance=REPORT_BOUNDARY_DISTANCE,
)
show_report(report_html)
